# 03 — Full RAG Pipeline: Query Expansion · Retrieval · Reranking · Generation

**Lab 05 · RAG with LangChain**

---

This notebook builds the complete RAG pipeline **step by step**, explaining
each component before assembling them into a single orchestrator.

| Step | Technique | Why |
|------|-----------|-----|
| 1 | **Query expansion** | One phrasing may miss relevant chunks; LLM variants broaden recall |
| 2 | **Multi-query retrieval** | Run similarity search for each variant, deduplicate results |
| 3 | **CrossEncoder reranking** | Bi-encoder retrieval is fast but approximate; CrossEncoder re-scores with full attention |
| 4 | **LCEL generation chain** | `prompt \| llm \| parser` — declarative, composable, provider-agnostic |

> **Prerequisites**: a valid API key for at least one LLM provider set in
> the `.env` file at the repo root (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`,
> or `GOOGLE_API_KEY`).

This notebook is **standalone** — no dependency on other lab modules.

## 0 — Setup: knowledge base and LLM

In [ ]:
import tempfile
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())  # loads .env from repo root

# ---------------------------------------------------------------------------
# LLM — change to any provider you have a key for
# "anthropic:claude-sonnet-4-6" | "openai:gpt-4o-mini" | "google_genai:gemini-2.5-flash"
# ---------------------------------------------------------------------------

MODEL_ID = "anthropic:claude-sonnet-4-6"
llm = init_chat_model(MODEL_ID)
print(f"LLM ready: {MODEL_ID}")

# ---------------------------------------------------------------------------
# Knowledge base — five short passages on distinct topics
# ---------------------------------------------------------------------------

PASSAGES = [
    (
        "transformer_architecture",
        "The Transformer architecture, introduced in 'Attention Is All You Need'"
        " (2017), replaced recurrent networks with a self-attention mechanism."
        " Self-attention allows each token to attend to every other token in the"
        " sequence simultaneously, enabling full parallelism during training."
        " The encoder-decoder design uses multi-head attention to capture"
        " different semantic relationships at multiple scales.",
    ),
    (
        "rag_overview",
        "Retrieval-Augmented Generation (RAG) combines a retrieval system with"
        " a generative language model. At query time, relevant passages are"
        " fetched from a vector store and injected into the LLM prompt as"
        " context. This grounds the model's response in factual documents,"
        " reducing hallucination and enabling knowledge updates without"
        " retraining the model.",
    ),
    (
        "cross_encoder",
        "A CrossEncoder takes a (query, passage) pair as a single input and"
        " produces a relevance score using full bidirectional attention across"
        " both texts. This is more accurate than bi-encoder similarity but"
        " much slower — O(n) inference calls per query. In practice,"
        " CrossEncoders are used as a reranker on a small candidate set"
        " retrieved by a fast bi-encoder (e.g. FAISS or ChromaDB).",
    ),
    (
        "query_expansion",
        "Query expansion generates alternative phrasings of a user question"
        " to improve retrieval recall. A single query may not match the exact"
        " vocabulary used in relevant documents. By generating 2-3 variants"
        " using an LLM and merging their retrieval results, the pipeline"
        " surfaces passages that would be missed by the original query alone.",
    ),
    (
        "lcel_chains",
        "LangChain Expression Language (LCEL) uses the pipe operator (|) to"
        " compose runnables: prompts, models, parsers, and retrievers. A chain"
        " like prompt | llm | StrOutputParser() is lazy — it builds a"
        " computation graph without executing it. LCEL chains support"
        " streaming, async, and batching transparently, and can be deployed"
        " with LangServe.",
    ),
]

# Build vectorstore
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)

docs = [
    Document(page_content=text, metadata={"topic": topic})
    for topic, text in PASSAGES
]
chunks = splitter.split_documents(docs)

DB_DIR = Path(tempfile.mkdtemp()) / "chroma_nb03"
vectorstore = Chroma(
    collection_name="rag_lab",
    embedding_function=embeddings,
    persist_directory=str(DB_DIR),
)
vectorstore.add_documents(chunks)
print(f"Indexed {vectorstore._collection.count()} chunks across {len(PASSAGES)} topics")

## Step 1 — Query expansion

A user's question is expressed in their own vocabulary, which may not match
the terminology in the indexed documents. Query expansion asks the LLM to
rephrase the question in 2 alternative ways, then retrieves for all variants.

**Implementation**: we use `with_structured_output()` to force the LLM to
return a typed Pydantic object instead of free text — this makes the output
reliably parseable without fragile regex.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


class QueryExpansion(BaseModel):
    questions: list[str] = Field(
        description="Alternative phrasings of the original question"
    )


EXPANSION_PROMPT = ChatPromptTemplate.from_template(
    "Generate {n} alternative phrasings of the following question to improve "
    "document retrieval. Return only the alternatives, not the original.\n\n"
    "Original question: {question}"
)


def expand_query(question: str, n: int = 2) -> list[str]:
    """Return the original question plus n LLM-generated alternatives."""
    structured_llm = llm.with_structured_output(QueryExpansion)
    result: QueryExpansion = structured_llm.invoke(
        EXPANSION_PROMPT.format_messages(question=question, n=n)
    )
    return [question] + result.questions


# Demo
question = "How does the attention mechanism work in transformers?"
expanded = expand_query(question)

print(f"Original : {expanded[0]}")
for i, alt in enumerate(expanded[1:], 1):
    print(f"Variant {i}: {alt}")

## Step 2 — Multi-query retrieval with deduplication

For each expanded query, we retrieve the top-k most similar chunks from
ChromaDB. The results are merged and deduplicated by the first 80 characters
of their content — a cheap fingerprint that avoids re-indexing identical chunks
from different queries.

In [ ]:
from langchain_core.documents import Document

RETRIEVAL_TOP_K = 5


def retrieve(questions: list[str]) -> list[Document]:
    """Retrieve and deduplicate documents for all query variants."""
    seen: set[str] = set()
    all_docs: list[Document] = []
    for q in questions:
        for doc in vectorstore.similarity_search(q, k=RETRIEVAL_TOP_K):
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)
    return all_docs


candidates = retrieve(expanded)

print(f"Retrieved {len(candidates)} unique chunks\n")
for i, doc in enumerate(candidates):
    print(f"[{i+1}] topic={doc.metadata.get('topic')}")
    print(f"     {doc.page_content[:100]}...")
    print()

## Step 3 — CrossEncoder reranking

The bi-encoder (BAAI/bge-m3) encodes query and document **independently** and
compares them via dot product. This is fast but approximate.

A **CrossEncoder** encodes the (query, document) pair **jointly**, letting
every query token attend to every document token. This is far more accurate
but requires one forward pass per candidate — hence it's applied only to the
small set retrieved in Step 2.

```
Bi-encoder:   embed(query) · embed(doc)    →  fast, O(1) at retrieval time
CrossEncoder: encode(query + doc)          →  accurate, O(n) per query
```

In [ ]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANK_TOP_K = 3

reranker = CrossEncoder(RERANKER_MODEL)


def rerank(
    query: str, documents: list[Document], top_k: int = RERANK_TOP_K
) -> list[Document]:
    """Re-score and sort documents by cross-encoder relevance."""
    if not documents:
        return []
    pairs = [(query, doc.page_content) for doc in documents]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(scores, documents, strict=True), key=lambda x: x[0], reverse=True
    )
    return [doc for _, doc in ranked][:top_k]


reranked = rerank(question, candidates)

print(f"After reranking — top {RERANK_TOP_K} chunks:\n")
for i, doc in enumerate(reranked):
    print(f"[{i+1}] topic={doc.metadata.get('topic')}")
    print(f"     {doc.page_content[:120]}...")
    print()

## Step 4 — LCEL generation chain

LangChain Expression Language uses `|` to compose **Runnables** into a
pipeline. Each component implements `.invoke()`, `.stream()`, and `.batch()`.

```
ChatPromptTemplate  →  fills {context} and {question} into a message list
        |
BaseChatModel       →  calls the LLM API, returns an AIMessage
        |
StrOutputParser     →  extracts the string content from the AIMessage
```

This chain is **lazy**: the `|` operator builds a graph, nothing executes
until `.invoke()` is called.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question using ONLY the context "
    "provided below. If the context does not contain enough information, say "
    '"I don\'t have enough information to answer this question."\n\n'
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

rag_chain = RAG_PROMPT | llm | StrOutputParser()

# Build context string from reranked chunks
context = "\n\n---\n\n".join(doc.page_content for doc in reranked)
answer = rag_chain.invoke({"context": context, "question": question})

print(f"Question: {question}\n")
print(f"Answer:\n{answer}")

## Step 5 — Full pipeline orchestrator

All four steps assembled into a single function — this is the production
equivalent of `app/query.py::answer_question()`.

In [ ]:
def rag_answer(user_question: str, verbose: bool = False) -> dict:
    """
    Full RAG pipeline:
      1. Query expansion   — LLM generates 2 alternative phrasings
      2. Retrieval         — similarity search for each variant, deduplicated
      3. Reranking         — CrossEncoder re-scores the candidate set
      4. Generation        — LCEL chain produces a grounded answer

    Returns {"answer": str, "sources": list[dict]}.
    """
    # 1. Query expansion
    questions = expand_query(user_question, n=2)
    if verbose:
        print(f"[1] Expanded to {len(questions)} queries:")
        for q in questions:
            print(f"    • {q}")

    # 2. Retrieval
    retrieved = retrieve(questions)
    if verbose:
        print(f"\n[2] Retrieved {len(retrieved)} unique chunks")

    if not retrieved:
        return {
            "answer": "I don't have enough information to answer this question.",
            "sources": [],
        }

    # 3. Reranking
    top_docs = rerank(user_question, retrieved)
    if verbose:
        print(f"[3] Reranked → top {len(top_docs)} chunks:")
        for doc in top_docs:
            print(f"    • {doc.metadata.get('topic')} — {doc.page_content[:60]}...")

    # 4. Generation
    ctx = "\n\n---\n\n".join(doc.page_content for doc in top_docs)
    answer_text = rag_chain.invoke({"context": ctx, "question": user_question})
    if verbose:
        print(f"\n[4] Answer generated ({len(answer_text)} chars)\n")

    sources = [
        {
            "topic": doc.metadata.get("topic"),
            "content": doc.page_content[:200],
        }
        for doc in top_docs
    ]
    return {"answer": answer_text, "sources": sources}

## Step 6 — Demo: run the full pipeline on several questions

In [ ]:
demo_questions = [
    "How does RAG reduce hallucinations in LLMs?",
    "Why is a CrossEncoder more accurate than a bi-encoder for reranking?",
    "What is LCEL and how do you compose chains with it?",
]

for q in demo_questions:
    print("=" * 70)
    result = rag_answer(q, verbose=True)
    print(f"\nAnswer:\n{result['answer']}")
    print(f"\nSources: {[s['topic'] for s in result['sources']]}")
    print()

## Step 7 — Key takeaways

| Component | This notebook | `app/query.py` equivalent |
|---|---|---|
| Query expansion | `expand_query()` + `QueryExpansion` Pydantic model | `expand_query()` |
| Multi-query retrieval | `retrieve()` with 80-char dedup key | inner loop in `answer_question()` |
| CrossEncoder reranking | `rerank()` via `sentence_transformers` | `rerank()` |
| LCEL chain | `RAG_PROMPT \| llm \| StrOutputParser()` | `build_rag_chain()` |
| Orchestrator | `rag_answer()` | `answer_question()` |

**Design notes:**
- `with_structured_output()` is more reliable than parsing free-text LLM output for query expansion
- The 80-char dedup key in `retrieve()` is a pragmatic heuristic — a full hash would be more robust but slower
- The CrossEncoder model (`ms-marco-MiniLM-L-6-v2`) is lightweight (~23 MB) and fast enough for top-10 reranking

**What comes next:**
- `04_rag_evaluation.ipynb` — measure quality with RAGAS (Context Precision, Recall, Faithfulness, Answer Relevancy)